## For the tested elements and variant within the 80K MPRA (cardiac neuro cava random)

In [55]:
from importlib import reload
import pandas as pd
import sys
import os
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)
import yaml

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [56]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


interesting_columns = [col_name, col_sequence, col_category, col_class, col_source,
                       col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

In [57]:
# load the data
input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
# split the metadata file headers by '#'
# Apply the function to each row and concatenate the results
pre_metadata_df_split = pd.concat(pre_metadata_df.apply(lambda row: hf.split_ids(row, id_col=col_name, separator='#'), axis=1).values)

# Reset the index
pre_metadata_df_split.reset_index(drop=True, inplace=True)

pre_metadata_df = pre_metadata_df_split.copy()
print(pre_metadata_df.shape[0]) # 80804
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))

# variant map
variant_map_path = config['variant_region_map']
variant_map = pd.read_csv(variant_map_path, sep="\t")
# variant_map.columns = ['ID', 'Region', 'REF', 'ALT']
# variant_map.to_csv('/home/kisa/coding/80K_MPRA/design_data/design_info/variant_region_map_new_colnames.tsv.gz', sep="\t", compression='gzip', index=False)
variant_map['tmp_label'] = variant_map['ID'].apply(hf.get_label)

# region bed
region_bed = config['region_bed']
region_bed = pd.read_csv(region_bed, sep="\t", header=None)
region_bed.columns = [f'region_{col_name}' for col_name in ['chr', 'start', 'end', 'name', 'score', 'strand']]

# vcf
vcf_path = config['variant_vcf']
vcf_df = pd.read_csv(vcf_path, sep='\t', comment="#", header=None)
vcf_df.columns = ['CHROM', 'var_pos', 'ID', 'vcf_REF', 'vcf_ALT', 'QUAL', 'FILTER', 'INFO']


80803


In [58]:
# helpful functions
def is_variant_related(name, variant_related_list):
    return name in variant_related_list

def is_reference_related(name, reference_related_list):
    return name in reference_related_list

def is_alternative_related(name, variant_related_list):
    return name in variant_related_list


# Combine columns
def combine_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith("_x"):
            base_name = col[:-2]  # Remove "_x"
            corresponding_y = base_name + "_y"
            # Combine _x and _y columns
            if corresponding_y in df.columns:
                new_columns[base_name] = df[col].fillna(df[corresponding_y])
            else:
                new_columns[base_name] = df[col]
        elif not col.endswith("_y"):  # Add columns that aren't paired with _x/_y
            new_columns[col] = df[col]
    return pd.DataFrame(new_columns)


def get_var_pos(row, var_pos_column, allele_column, seq_start_column, variant_1_based=True, start_0_based=True):
    """
    Computes the variant pos (0-based coordinate of variant in the string)
    for the alternative sequences with different cased of the data
    (variant_1_based and start_0_based is the default)

    If variant is 0 based set variant_1_based to false (same goes for start)
    """
    variant_position = 'NA'
    if hf.is_alternative(row[allele_column]):
        variant_0_based = row[var_pos_column] - 1 if variant_1_based else row[var_pos_column]
        seq_start_0_based = row[seq_start_column] if start_0_based else row[seq_start_column] - 1
        variant_position = variant_0_based - seq_start_0_based
    return variant_position

import re
def find_indel_pattern(row, ref_column, alt_column):
    """Check if in ref or alt is more than 1 subsequent nucleotide indicating an indel
    Special case: I want it to check for the notation of muliallelic variants as well (A,T) if any of these is an indel"""
    if hf.is_alternative(row[col_allele]):
        pattern = r'(^[ACGT]{2,})|(,[ACGT]{2,})'
        # Check if the text matches the pattern
        ref_indel = bool(re.match(pattern, row[ref_column]))
        alt_indel = bool(re.match(pattern, row[alt_column]))
        return ref_indel or alt_indel
    else: False


def get_variant_alternative(row, col_sequence, col_variant_pos, col_allele, col_variant_class='variant_class'):
    """Return the char at the variant pos position"""
    if not hf.is_alternative(row[col_allele]):
        return 'NA'
    if row[col_variant_class] == 'SNV':
        variant_position = int(row[col_variant_pos])
        return row[col_sequence][variant_position]
    elif row[col_variant_class] == 'indel':
        if ',' in row['vcf_ALT']:
            raise ValueError('Special case of indel. Please check manually')
        return row['vcf_ALT']
    return 'NA'


# dict of chr number to refseq chromosome number
chrom_2_refseq = {"chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'

In [59]:
variant_region_map = variant_map.merge(region_bed, left_on='Region', right_on='region_name', how='inner')
variant_region_map

,ID,Region,REF,ALT,tmp_label,region_chr,region_start,region_end,region_name,region_score,region_strand
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2191262,2191532,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2191971,2192241,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2192249,2192519,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,cardiac_neuro_cava_random,chr1,2192936,2193206,cardiac_neuro_cava_random:SKI|ENSG00000157933....,.,+
...,...,...,...,...,...,...,...,...,...,...,...
46816,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274993,109275263,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+
46817,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109275105,109275375,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+
46818,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109274993,109275263,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+
46819,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher,chr1,109275105,109275375,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,.,+


### Subset of data for the tested group

In [60]:
group_name = 'cardiac_neuro_cava_random'
pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
expected_number = pre_metadata_df_group.shape[0]
variant_region_df_group = variant_region_map.loc[variant_region_map['tmp_label'] == group_name].copy()
region_bed_group = region_bed.loc[region_bed['region_name'].str.contains(group_name)].copy()
vcf_df_group = vcf_df.loc[vcf_df['ID'].str.contains(group_name)].copy()

In [61]:
print('Expected number: ', expected_number)

Expected number:  73940


In [62]:
# add the columns of the metadata file
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])

# if variant related or element
variant_related_list = set(variant_region_df_group['REF'].to_list()).union(set(variant_region_df_group['ALT'].to_list()))
pre_metadata_df_group[col_category] = pre_metadata_df_group[col_name].apply(lambda name: 'variant' if is_variant_related(name, variant_related_list) else 'element')


pre_metadata_df_group[col_class] = pre_metadata_df_group[col_name].apply(lambda name: 'variant negative control' if is_variant_related(name, variant_related_list) else 'element inactive control')
pre_metadata_df_group[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group[col_ref] = 'GRCh38'

# add allele
alternative_related_list = set(variant_region_df_group['ALT'].to_list())
reference_related_list = set(variant_region_df_group['REF'].to_list())
pre_metadata_df_group[col_allele] = pre_metadata_df_group[col_name].apply(lambda name: 'alt' if is_alternative_related(name, alternative_related_list) else 'ref' if is_reference_related(name, reference_related_list) else 'NA')

pre_metadata_df_group[col_variant_class] = 'NA'
pre_metadata_df_group[col_variant_pos] = 'NA'
pre_metadata_df_group[col_SPDI] = 'NA'
pre_metadata_df_group[col_info] = ''

### Focus on Elements

In [63]:
pre_metadata_df_group_element = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'element'].copy()
pre_metadata_df_group_element.name.shape[0]

8994

In [64]:
region_bed.columns

Index(['region_chr', 'region_start', 'region_end', 'region_name',
       'region_score', 'region_strand'],
      dtype='object')

In [65]:
pre_metadata_df_group_element_region = pre_metadata_df_group_element.merge(region_bed, left_on=col_name, right_on='region_name', how='inner')

In [66]:
pre_metadata_df_group_element_region.shape[0] # 8900

8900

Not all elements matched the region bed => investigate these further
- if long or small the basename can be found in the region bed 
  - e.g. cardiac_neuro_cava_random:ALT_DEAF1|ENSG00000177030.19|EH38E2937977,SLC25A22|ENSG00000177542.11|EH38E2937977_rev_tile1-1_DEAF1|ENSG00000177030.19|EH38E2937977|11-744461-A-C,SLC25A22|ENSG00000177542.11|EH38E2937977|11-744461-A-C => `cardiac_neuro_cava_random:DEAF1|ENSG00000177030.19|EH38E2937977,SLC25A22|ENSG00000177542.11|EH38E2937977_rev_tile1-1` is in the region.bed (remove :REF_ or :ALT and only print name_rev/fwd_tileInfo)

In [67]:
elements_matched = set(pre_metadata_df_group_element_region[col_name].to_list())
all_elements = set(pre_metadata_df_group_element[col_name].to_list())
elements_not_matched = all_elements - elements_matched

In [68]:
count = 1
for elem in elements_not_matched:
    print(elem)
    count += 1
    if count > 5:
        break

cardiac_neuro_cava_random:ALT_ELAVL3|ENSG00000196361.10|EH38E3290526_rev_tile1-1_ELAVL3|ENSG00000196361.10|EH38E3290525|19-11453196-G-C
cardiac_neuro_cava_random:ALT_DNMT3A|ENSG00000119772.19|EH38E3331372_rev_tile1-1_DNMT3A|ENSG00000119772.19|EH38E3331373|2-25230797-G-C
cardiac_neuro_cava_random:ALT_RERE|ENSG00000142599.20|EH38E2783640_rev_tile1-1_RERE|ENSG00000142599.20|EH38E2783640|1-8313520-T-G
cardiac_neuro_cava_random:ALT_BCL10|ENSG00000142867.14|EH38E2822216_rev_tile1-1_BCL10|ENSG00000142867.14|EH38E2822216|1-85320523-A-C
cardiac_neuro_cava_random:ALT_ZMYND8|ENSG00000101040.20|EH38E3436249_rev_tile1-1_ZMYND8|ENSG00000101040.20|EH38E3436249|20-47334605-A-T


In [69]:
# e.g. cardiac_neuro_cava_random:ALT_AHDC1|ENSG00000126705.15|EH38E2797860_rev_tile1-1_AHDC1|ENSG00000126705.15|EH38E2797860|1-27540408-A-C
###### cardiac_neuro_cava_random:AHDC1|ENSG00000126705.15|EH38E2797860_rev_tile1-1 can be matched in the region file
# example_string = 'cardiac_neuro_cava_random:ALT_HADH|ENSG00000138796.18|EH38E3599019_fwd_tile1-1_HADH|ENSG00000138796.18|EH38E3599019|4-108037975-A-G'

# example_string_split = example_string.split('_')
# f'cardiac_neuro_cava_random:{example_string_split[4]}_{example_string_split[5]}_{example_string_split[6]}'

In [70]:
def prepare_matchable_region_name(name):
    """
    Some elements are not matchable anymore with the region map => rename them
    """
    if ':REF_' in name or ':ALT_' in name:
        name_split = name.split('_')
        return f'cardiac_neuro_cava_random:{name_split[4]}_{name_split[5]}_{name_split[6]}'
    return name

In [71]:
pre_metadata_df_group_element['matchable_region_name'] = pre_metadata_df_group_element[col_name].apply(prepare_matchable_region_name)

In [72]:
pre_metadata_df_group_element_region = pre_metadata_df_group_element.merge(region_bed, left_on='matchable_region_name', right_on='region_name', how='inner')
pre_metadata_df_group_element_region.drop(columns=['region_name', 'matchable_region_name'], inplace=True)
pre_metadata_df_group_element_region.columns = [col.split('region_')[1] if 'region_' in col else col for col in pre_metadata_df_group_element_region.columns]

In [73]:
print('Now all the elements could be matched to the regions bed: ',pre_metadata_df_group_element_region.shape[0])

Now all the elements could be matched to the regions bed:  8994


In [74]:
pre_metadata_df_group_element_region_final = pre_metadata_df_group_element_region[interesting_columns].copy()
pre_metadata_df_group_element_region_final[col_sequence].nunique()

8994

In [75]:
pre_metadata_df_group_element_region_final.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info'],
      dtype='object')

### Focus on variants

In [76]:
pre_metadata_df_group_variant = pre_metadata_df_group.loc[pre_metadata_df_group[col_category] == 'variant'].copy()
print(pre_metadata_df_group_variant.shape[0])
pre_metadata_df_group_variant[col_name].nunique()

64946


64946

In [77]:
pre_metadata_df_group_region_alt = pre_metadata_df_group_variant.merge(variant_region_map[['ID', 'ALT', 'region_chr', 'region_start', 'region_end', 'region_strand']], left_on=col_name, right_on='ALT', how='left')
print(pre_metadata_df_group_region_alt.shape[0])
print(pre_metadata_df_group_region_alt[col_name].nunique())
only_reference_sequences = variant_region_map[['ID', 'REF', 'region_chr', 'region_start', 'region_end', 'region_strand']].drop_duplicates()
pre_metadata_df_group_region_alt_ref = pre_metadata_df_group_region_alt.merge(only_reference_sequences, left_on=col_name, right_on='REF', how='left')
print(pre_metadata_df_group_region_alt_ref.shape[0])
print(pre_metadata_df_group_region_alt_ref[col_name].nunique())

64946
64946
92748
64946


as you can see in the output 92748 is the number of rows after joining the reference information => duplicates are in the file
- first combine the _x and _y columns and remove them later

#### Combine the columns _x, _y from joining

In [78]:
# Apply the function
pre_metadata_df_group_region_alt_ref = combine_columns(pre_metadata_df_group_region_alt_ref)

In [79]:
pre_metadata_df_group_region_alt_ref
pre_metadata_df_group_region_alt_ref.rename(columns=lambda x: x.replace('region_', '') if 'region_' in x else x , inplace=True)
# Converting float columns to integers
pre_metadata_df_group_region_alt_ref['start'] = pre_metadata_df_group_region_alt_ref['start'].astype(int)
pre_metadata_df_group_region_alt_ref['end'] = pre_metadata_df_group_region_alt_ref['end'].astype(int)


In [80]:
# remove duplicates:
pre_metadata_df_group_region_alt_ref_dedup = pre_metadata_df_group_region_alt_ref.drop_duplicates(subset=interesting_columns)
pre_metadata_df_group_region_alt_ref_dedup.shape[0]

64946

In [81]:
# split by ids:
# Apply the function to each row and concatenate the results
vcf_df_group_split = pd.concat(vcf_df_group.apply(lambda row: hf.split_ids(row, id_col='ID', separator=';'), axis=1).values)

# Reset the index
vcf_df_group_split.reset_index(drop=True, inplace=True)
print(vcf_df_group.shape[0])
print(vcf_df_group_split.shape[0]) # 45745 before 45737

45737
45745


In [82]:
variant_pre_metadata_vcf = pre_metadata_df_group_region_alt_ref_dedup.merge(vcf_df_group_split[['ID', 'var_pos', 'vcf_REF', 'vcf_ALT']], on='ID', how='left')
variant_pre_metadata_vcf.shape[0] # 64946

64946

In [83]:
variant_pre_metadata_vcf

,name,sequence,tmp_label,category,class,source,ref,allele,variant_class,variant_pos,...,ID,ALT,chr,start,end,strand,REF,var_pos,vcf_REF,vcf_ALT
0,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,GTCCCAGCTCCCCACTGATGTGAAAGGTGGTGGTGAGTTAACAGCT...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,NaN,chr1,2179507,2179777,+,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,2179591,T,C
1,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,CCTGATCTGCCCTGTCCGTGACGCTTCTGCTCAGTAGCTGAGCACG...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,NaN,chr1,2191262,2191532,+,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,2191444,G,A
2,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,CCTCTGGGTGACCCGGAGAACACCAAGGCTGTGAGAAATGGGAGGC...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,NaN,chr1,2191971,2192241,+,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,2192015,G,T
3,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,CCATGCGGTGGCCACAGCCTCGGGTGAGTTCCGGTTCCAAAGTACC...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,NaN,chr1,2192249,2192519,+,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,2192366,T,G
4,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,GGACTCCGGTGCCTTCGCATTCCCGAGCTGTTTTTGCTTCTGGAAG...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,ref,NA,NA,...,cardiac_neuro_cava_random:SKI|ENSG00000157933....,NaN,chr1,2192936,2193206,+,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,2193142,G,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64941,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,GGAGCTCTGCCTCACCCCACCTGGCCCCAATTGTCCAGCTTGTAGA...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,chrX,154545000,154545270,-,NaN,154545206,A,G
64942,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,ATGTCTGAATTCACCTCCAAATAATGGGAAAACTCCTAGGTATATA...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,chrX,154549802,154550072,-,NaN,154549923,T,G
64943,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCCC...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,chrX,154552145,154552415,-,NaN,154552289,C,T
64944,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,CCTCTGCCCTCCCTGGCTTCTTCCCCTGTCCCTCCTTTCCCTTCTC...,cardiac_neuro_cava_random,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,alt,NA,NA,...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,chrX,154552145,154552415,-,NaN,154552371,G,A


In [84]:
# check if the matching was sucessfull
pre_metadata_df_group_region_alt_ref_dedup_names = set(pre_metadata_df_group_region_alt_ref_dedup['ID'].to_list())
vcf_df_group_names = set(vcf_df_group_split['ID'].to_list())
pre_metadata_df_group_region_alt_ref_dedup_names - vcf_df_group_names

set()

#### Compute variant position
- is the start 0-based? - yes
- is the variant position 1-based? chr20:50,871,409-50,871,411 (G) yes (https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr20%3A50871409%2D50871411&hgsid=2392258313_huE5stAQaQkB4376zP5rx9IaAEMn)

In [85]:
variant_pre_metadata_vcf[col_variant_pos] = variant_pre_metadata_vcf.apply(lambda row: get_var_pos(row, var_pos_column='var_pos', allele_column=col_allele, seq_start_column=col_start, variant_1_based=True, start_0_based=True), axis=1)
variant_pre_metadata_vcf[[col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele]]

# add variant_class: SNV or indel
variant_pre_metadata_vcf["is_indel"] = variant_pre_metadata_vcf.apply(lambda row: find_indel_pattern(row, "vcf_REF", "vcf_ALT"), axis=1)
variant_pre_metadata_vcf[col_variant_class] = variant_pre_metadata_vcf['is_indel'].apply(lambda indel: 'indel' if indel else 'SNV')
variant_pre_metadata_vcf[[col_name, col_sequence, 'var_pos', col_chr, col_start, col_end, 'variant_pos', col_allele, 'vcf_REF', 'vcf_ALT', col_variant_class]]
variant_pre_metadata_vcf[col_variant_class].value_counts() # no indel

variant_class
SNV    64946
Name: count, dtype: int64

#### Sometimes the vcf file contains colons in the ALT column so we take the char at the variant position

In [86]:
variant_pre_metadata_vcf['real_ALT'] = variant_pre_metadata_vcf.apply(lambda row: get_variant_alternative(row, col_sequence=col_sequence, col_variant_pos=col_variant_pos, col_allele=col_allele), axis=1)

#### Add SPDI

In [87]:
# add SPDI for alt
variant_pre_metadata_vcf['SPDI'] = variant_pre_metadata_vcf.apply(lambda row: create_speedy_chromosomes(f"{row['chr']}-{row['var_pos']}-{row['vcf_REF']}-{row['real_ALT']}") if hf.is_alternative(row[col_allele]) else 'NA', axis=1)

# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = variant_region_df_group.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = variant_pre_metadata_vcf.loc[variant_pre_metadata_vcf[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict

# function to add a list of SPDI values from the REF to the metadata table
def add_spdi_values_2_reference(row):
    if hf.is_reference(row[col_allele]):
        # check if row[col_name] is in ref_alt_dict
        if not row[col_name] in ref_alt_dict:
            # raise exception
            raise ValueError('Reference ID not found in ref_alt_dict')
        row[col_SPDI] = [alt_spdi_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        row[col_allele] = ['ref' for _ in ref_alt_dict[row[col_name]]]
    return row

variant_pre_metadata_vcf = variant_pre_metadata_vcf.apply(add_spdi_values_2_reference, axis=1)
variant_pre_metadata_vcf_final = variant_pre_metadata_vcf[interesting_columns].copy()

In [88]:
pre_metadata_df_group_element_region_final.columns

Index(['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr',
       'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI',
       'allele', 'info'],
      dtype='object')

In [89]:
pre_metadata_df_group_element_region_final[col_name].nunique()


8994

In [90]:
pre_metadata_df_group_element_region_final[col_sequence].nunique()

8994

In [91]:
variant_pre_metadata_vcf_final[col_name].nunique()

64946

In [92]:
variant_pre_metadata_vcf_final[col_sequence].nunique()

64946

In [93]:
### Merge variant and element results
combined_metadata_final = pd.concat([pre_metadata_df_group_element_region_final, variant_pre_metadata_vcf_final], ignore_index=True)

In [94]:
print('expected number: ', expected_number)
print('number of rows in the metadata file: ', combined_metadata_final.shape[0])
combined_metadata_final[col_sequence].nunique()

expected number:  73940
number of rows in the metadata file:  73940


73940

In [95]:
duplicated = combined_metadata_final.loc[combined_metadata_final.duplicated(subset=[col_name], keep=False)]
duplicated

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info


In [97]:
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
combined_metadata_final[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0